# Libraries

In [1]:
import pandas as pd
import numpy as np

import statistics

from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Dataset preparing

In [2]:
dataset = "datasets/Nursery.csv"
def print_bad_lines(line):
    print(f"Bad line: {line}")

df = pd.read_csv(dataset, on_bad_lines=print_bad_lines, engine='python')

# Define the categories with the desired order
categories = [
    ['usual', 'pretentious', 'great_pret'],  # parents
    ['proper', 'less_proper', 'improper', 'critical', 'very_crit'],  # has_nurs
    ['complete', 'completed', 'incomplete', 'foster'],  # form
    ['1', '2', '3', 'more'],  # children
    ['convenient', 'less_conv', 'critical'],  # housing
    ['convenient', 'inconv'],  # finance
    ['nonprob', 'slightly_prob', 'problematic'],  # social
    ['recommended', 'priority', 'not_recom']  # health
]

# Initialize the OrdinalEncoder with the specified categories
ordinal_encoder = OrdinalEncoder(categories=categories)

classes = df.pop('class')
columns = df.columns

# Fit and transform the data
df = ordinal_encoder.fit_transform(df)

# Convert the result back to a DataFrame for better readability
df = pd.DataFrame(df, columns=columns)
df['class'] = classes
df = df[df['class'] != 'recommend']  # Remove rows where class is 'recommend'

In [3]:
df

,parents,has_nurs,form,children,housing,finance,social,health,class
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,priority
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,not_recom
4,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,priority
5,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,not_recom
6,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,priority
...,...,...,...,...,...,...,...,...,...
12955,2.0,4.0,3.0,3.0,2.0,1.0,1.0,1.0,spec_prior
12956,2.0,4.0,3.0,3.0,2.0,1.0,1.0,2.0,not_recom
12957,2.0,4.0,3.0,3.0,2.0,1.0,2.0,0.0,spec_prior
12958,2.0,4.0,3.0,3.0,2.0,1.0,2.0,1.0,spec_prior


# Defining easy and hard subclasses for experiment

In [4]:
auc_df = pd.read_csv('processed_results_nursery.csv')
auc_df

,Positive,Negative,Easy,Hard,AUC,AUC_Easy,AUC_Hard
0,['not_recom'],"['priority', 'very_recom']",['very_recom'],['priority'],1.000000,1.000000,1.000000
1,['not_recom'],"['priority', 'very_recom']",['very_recom'],['priority'],1.000000,1.000000,1.000000
2,['not_recom'],"['priority', 'very_recom']",['priority'],['very_recom'],1.000000,1.000000,1.000000
3,['not_recom'],"['priority', 'very_recom']",['very_recom'],['priority'],1.000000,1.000000,1.000000
4,['not_recom'],"['priority', 'very_recom']",['priority'],['very_recom'],1.000000,1.000000,1.000000
...,...,...,...,...,...,...,...
124995,['priority'],"['very_recom', 'spec_prior']",['very_recom'],['spec_prior'],0.898939,0.949360,0.950148
124996,['priority'],"['very_recom', 'spec_prior']",['very_recom'],['spec_prior'],0.898939,0.949360,0.950148
124997,['priority'],"['very_recom', 'spec_prior']",['spec_prior'],['very_recom'],0.898939,0.950148,0.949360
124998,['priority'],"['very_recom', 'spec_prior']",['spec_prior'],['very_recom'],0.898939,0.950148,0.949360


In [5]:
auc_df['diff_AUC'] = abs(auc_df['AUC_Easy'] - auc_df['AUC_Hard'])
auc_df.sort_values(by='diff_AUC', ascending=False, inplace=True)
auc_df

,Positive,Negative,Easy,Hard,AUC,AUC_Easy,AUC_Hard,diff_AUC
47775,['spec_prior'],"['priority', 'not_recom', 'very_recom']",['very_recom'],"['priority', 'not_recom']",0.885614,1.000000,0.883083,0.116917
98449,['spec_prior'],"['not_recom', 'very_recom', 'priority']","['not_recom', 'priority']",['very_recom'],0.885614,0.883083,1.000000,0.116917
47761,['spec_prior'],"['priority', 'not_recom', 'very_recom']","['priority', 'not_recom']",['very_recom'],0.885614,0.883083,1.000000,0.116917
47762,['spec_prior'],"['priority', 'not_recom', 'very_recom']","['priority', 'not_recom']",['very_recom'],0.885614,0.883083,1.000000,0.116917
47771,['spec_prior'],"['priority', 'not_recom', 'very_recom']",['very_recom'],"['priority', 'not_recom']",0.885614,1.000000,0.883083,0.116917
...,...,...,...,...,...,...,...,...
9,['not_recom'],"['priority', 'very_recom']",['very_recom'],['priority'],1.000000,1.000000,1.000000,0.000000
10,['not_recom'],"['priority', 'very_recom']",['very_recom'],['priority'],1.000000,1.000000,1.000000,0.000000
11,['not_recom'],"['priority', 'very_recom']",['priority'],['very_recom'],1.000000,1.000000,1.000000,0.000000
12,['not_recom'],"['priority', 'very_recom']",['priority'],['very_recom'],1.000000,1.000000,1.000000,0.000000


In [6]:
auc_df[auc_df['diff_AUC'] > 0.11]

,Positive,Negative,Easy,Hard,AUC,AUC_Easy,AUC_Hard,diff_AUC
47775,['spec_prior'],"['priority', 'not_recom', 'very_recom']",['very_recom'],"['priority', 'not_recom']",0.885614,1.000000,0.883083,0.116917
98449,['spec_prior'],"['not_recom', 'very_recom', 'priority']","['not_recom', 'priority']",['very_recom'],0.885614,0.883083,1.000000,0.116917
47761,['spec_prior'],"['priority', 'not_recom', 'very_recom']","['priority', 'not_recom']",['very_recom'],0.885614,0.883083,1.000000,0.116917
47762,['spec_prior'],"['priority', 'not_recom', 'very_recom']","['priority', 'not_recom']",['very_recom'],0.885614,0.883083,1.000000,0.116917
47771,['spec_prior'],"['priority', 'not_recom', 'very_recom']",['very_recom'],"['priority', 'not_recom']",0.885614,1.000000,0.883083,0.116917
...,...,...,...,...,...,...,...,...
98407,['spec_prior'],"['not_recom', 'very_recom', 'priority']",['very_recom'],"['not_recom', 'priority']",0.885614,1.000000,0.883083,0.116917
98439,['spec_prior'],"['not_recom', 'very_recom', 'priority']",['very_recom'],"['priority', 'not_recom']",0.885614,1.000000,0.883083,0.116917
47778,['spec_prior'],"['priority', 'not_recom', 'very_recom']",['very_recom'],"['not_recom', 'priority']",0.885614,1.000000,0.883083,0.116917
47787,['spec_prior'],"['priority', 'not_recom', 'very_recom']","['not_recom', 'priority']",['very_recom'],0.885614,0.883083,1.000000,0.116917


In [7]:
experiment_setup = auc_df.iloc[0]
experiment_setup

Positive                             ['spec_prior']
Negative    ['priority', 'not_recom', 'very_recom']
Easy                                 ['very_recom']
Hard                      ['priority', 'not_recom']
AUC                                        0.885614
AUC_Easy                                        1.0
AUC_Hard                                   0.883083
diff_AUC                                   0.116917
Name: 47775, dtype: object

# Quantifiers

## Utils

In [8]:
def getTPRandFPRbyThreshold(validation_scores):
    unique_scores = np.arange(0.01, 1.00, 0.01)
    arrayOfTPRandFPRByTr = []

    total_positive = np.sum(validation_scores[:, 2] == 1)
    total_negative = np.sum(validation_scores[:, 2] == 2)

    for threshold in unique_scores:
        fp = np.sum((validation_scores[:, 0] > threshold) & (validation_scores[:, 2] == 2))
        tp = np.sum((validation_scores[:, 0] > threshold) & (validation_scores[:, 2] == 1))
        tpr = tp / total_positive if total_positive > 0 else 0
        fpr = fp / total_negative if total_negative > 0 else 0

        arrayOfTPRandFPRByTr.append([round(threshold, 2), tpr, fpr])

    return np.array(arrayOfTPRandFPRByTr, dtype=object)

In [9]:
def DySyn_distance(x, method="hellinger"):
    if method == "ord":
        x_dif = x[0, :] - x[1, :]
        acum = 0
        aux = 0
        for val in x_dif:
            aux += val
            acum += aux
        return abs(acum)

    if method == "topsoe":
        return sum(x[0, i] * np.log((2 * x[0, i]) / (x[0, i] + x[1, i])) +
                   x[1, i] * np.log((2 * x[1, i]) / (x[1, i] + x[0, i])) for i in range(x.shape[1]))

    if method == "jensen_difference":
        return sum(((x[0, i] * np.log(x[0, i]) + x[1, i] * np.log(x[1, i])) / 2) -
                   ((x[0, i] + x[1, i]) / 2) * np.log((x[0, i] + x[1, i]) / 2) for i in range(x.shape[1]))

    if method == "taneja":
        return sum(((x[0, i] + x[1, i]) / 2) * np.log((x[0, i] + x[1, i]) / (2 * np.sqrt(x[0, i] * x[1, i])))
                   for i in range(x.shape[1]))

    if method == "hellinger":
        return 2 * np.sqrt(1 - sum(np.sqrt(x[0, i] * x[1, i]) for i in range(x.shape[1])))

    if method == "prob_symm":
        return 2 * sum(((x[0, i] - x[1, i]) ** 2) / (x[0, i] + x[1, i]) for i in range(x.shape[1]))

    raise ValueError("measure argument must be a valid option")

def getHist(scores, nbins):
    breaks = np.linspace(0, 1, nbins + 1)
    breaks[-1] = 1.1
    re = np.full(len(breaks) - 1, 1 / (len(breaks) - 1))

    for i in range(1, len(breaks)):
        re[i - 1] = (re[i - 1] + np.sum((scores >= breaks[i - 1]) & (scores < breaks[i]))) / (len(scores) + 1)

    return re

def TernarySearch(left, right, f, eps=1e-4):
    while True:
        if abs(left - right) < eps:
            return (left + right) / 2

        leftThird = left + (right - left) / 3
        rightThird = right - (right - left) / 3

        if f(leftThird) > f(rightThird):
            left = leftThird
        else:
            right = rightThird

In [10]:
def MoSS(n, alpha, m):
    p_score = np.random.uniform(size=int(n * alpha)) ** m
    n_score = 1 - (np.random.uniform(size=int(round(n * (1 - alpha), 0))) ** m)
    scores = np.column_stack((np.concatenate((p_score, n_score)), np.concatenate((p_score, n_score)), np.concatenate((np.ones(len(p_score)), np.full(len(n_score), 2)))))
    # pdb.set_trace()
    return scores

## Classic quantifiers

In [11]:
def CC(test, thr=0.5):
    result = np.sum(test >= thr) / len(test)
    result = max(0, min(result, 1))

    return np.array([result, 1 - result], dtype=float)

def ACC(test, TprFpr, thr=0.5):
    dC = CC(test)  # Implement CC function separately

    tpr_fpr_row = TprFpr[TprFpr[:, 0] == thr, 1:3].astype(float)
    if tpr_fpr_row.size == 0:
        raise ValueError("Threshold value not found in TprFpr.")

    tpr, fpr = tpr_fpr_row[0]

    result = (dC[0] - fpr) / (tpr - fpr) if (tpr - fpr) != 0 else 0
    result = max(0, min(result, 1))

    return np.array([result, 1 - result], dtype=float)

def DyS(p_score, n_score, test, measure="topsoe", bins=np.arange(2, 22, 2), err=1e-5):
    results = []

    for b_size in bins:
        Sty_1 = getHist(p_score, b_size)  # Implement getHist separately
        Sty_2 = getHist(n_score, b_size)
        Uy = getHist(test, b_size)

        def f(x):
            return DySyn_distance(np.vstack([(Sty_1 * x) + (Sty_2 * (1 - x)), Uy]), method=measure)  # Implement DyS_distance separately

        best_alpha = TernarySearch(0, 1, f, err)  # Implement TernarySearch separately
        results.append(best_alpha)

    result = statistics.median(results)
    result = max(0, min(result, 1))

    return np.array([result, 1 - result])

In [12]:
def X(ts, TprFpr):
    dC = CC(ts)  # Implement CC function separately

    min_index = abs((1 - TprFpr[:, 1]) - TprFpr[:, 2])
    min_index = np.argmin(min_index)

    tpr_fpr_row = TprFpr[min_index, 1:3].astype(float)
    if tpr_fpr_row.size == 0:
        raise ValueError("Threshold value not found in TprFpr.")

    tpr, fpr = tpr_fpr_row

    result = (dC[0] - fpr) / (tpr - fpr) if (tpr - fpr) != 0 else 0
    result = max(0, min(result, 1))

    return np.array([result, 1 - result], dtype=float)

In [13]:
def MAX(ts, TprFpr):
    dC = CC(ts)  # Implement CC function separately

    # Usual MAX implementation
    max_index = abs(TprFpr[:, 1] - TprFpr[:, 2])
    max_index = np.argmax(max_index)

    tpr_fpr_row = TprFpr[max_index, 1:3].astype(float)

    tpr, fpr = tpr_fpr_row

    result = (dC[0] - fpr) / (tpr - fpr) if (tpr - fpr) != 0 else 0
    result = max(0, min(result, 1))

    return np.array([result, 1 - result], dtype=float)

In [14]:
def T50(ts, TprFpr):
    dC = CC(ts)  # Implement CC function separately

    # Usual T50 implementation
    min_index = abs(TprFpr[:, 1] - 0.5)
    min_index = np.argmin(min_index)

    tpr_fpr_row = TprFpr[min_index, 1:3].astype(float)

    tpr, fpr = tpr_fpr_row

    result = (dC[0] - fpr) / (tpr - fpr) if (tpr - fpr) != 0 else 0
    result = max(0, min(result, 1))

    return np.array([result, 1 - result], dtype=float)

In [15]:
def MS(ts, TprFpr):
  results = []
  dC = CC(ts)  # Implement CC function separately

  # Usual MS implementation
  threshold_set = np.arange(0.01, 1.00, 0.01)

  for threshold in threshold_set:
    threshold = round(threshold, 2) # 0.060000003 shenanigans
    tpr, fpr = TprFpr[TprFpr[:, 0] == threshold, 1:3].astype(float)[0]

    result = (dC[0] - fpr) / (tpr - fpr) if (tpr - fpr) != 0 else 0
    result = max(0, min(result, 1))

    results.append(result)

    result = np.median(results)
  return np.array([result, 1 - result], dtype=float)

In [16]:
def MS2(ts, TprFpr):
    results = []

    dC = CC(ts)  # Implement CC function separately

    # Usual MS2 implementation
    index = np.where(abs(TprFpr[:,1]-TprFpr[:,2]) > (1/4))[0].tolist()
    threshold_set = TprFpr[index,0]
    if threshold_set.shape[0] == 0:
      threshold_set = np.arange(0.01, 1.00, 0.01)

    # else usual MS
    for threshold in threshold_set:
      threshold = round(threshold, 2) # 0.060000003 shenanigans
      tpr, fpr = TprFpr[TprFpr[:, 0] == threshold, 1:3].astype(float)[0]

      result = (dC[0] - fpr) / (tpr - fpr) if (tpr - fpr) != 0 else 0
      result = max(0, min(result, 1))

      results.append(result)

    result = np.median(results)
    return np.array([result, 1 - result], dtype=float)

In [17]:
def SMM(p_scores, n_scores, t_scores):
  mean_p_scores = np.mean(p_scores)
  mean_n_scores = np.mean(n_scores)
  mean_t_scores = np.mean(t_scores)

  alpha = (mean_t_scores - mean_n_scores) / (mean_p_scores - mean_n_scores)
  alpha = max(0, min(alpha, 1))

  return np.round([alpha, abs(1-alpha)], 2)

## Synthetic quantifiers

In [18]:
def DySyn(ts, measure, MF=np.arange(0.1, 1.0, 0.2)):
    MF = np.round(MF, 2)

    results = []
    distances = []

    for mf in MF:
        scores = MoSS(1000, 0.5, mf)  # Implement MoSS function separately
        test_p = scores[scores[:, 2] == 1, 0]
        test_n = scores[scores[:, 2] == 2, 0]

        if measure == "sord":
            rQnt = DySyn_SORD(test_p, test_n, ts)  # Implement DySyn_SORD separately
        else:
            rQnt = DySyn_DyS(test_p, test_n, ts, measure, [10])  # Implement DySyn_DyS separately

        distances.append(rQnt[1])
        results.append(rQnt[0][0])

    best_result = round(results[np.argmin(distances)], 2)
    return [np.array([best_result, 1 - best_result]), min(distances), MF[np.argmin(distances)]]

def DySyn_DyS(p_score, n_score, test, measure="hellinger", b_sizes = list(range(2, 21, 2)) + [30]):
    results = []
    vDistAll = []

    for b_size in b_sizes:
        Sty_1 = getHist(p_score, b_size)  # Implement getHist separately
        Sty_2 = getHist(n_score, b_size)
        Uy = getHist(test, b_size)

        def f(x):
            return DySyn_distance(np.vstack([(Sty_1 * x) + (Sty_2 * (1 - x)), Uy]), method=measure)  # Implement DySyn_distance separately

        best_alpha = TernarySearch(0, 1, f, 1e-2)  # Implement TernarySearch separately
        results.append(best_alpha)
        vDistAll.append(f(best_alpha))

    median_result = np.median(results)
    return [np.array([round(median_result, 2), 1 - round(median_result, 2)]), min(vDistAll)]

def PNTDiff(pos, neg, test, pos_prop):
    p_w = pos_prop / len(pos)
    n_w = (1 - pos_prop) / len(neg)
    t_w = -1 / len(test)

    p = np.column_stack((pos, np.full(len(pos), p_w)))
    n = np.column_stack((neg, np.full(len(neg), n_w)))
    t = np.column_stack((test, np.full(len(test), t_w)))

    v = np.vstack((p, n, t))
    v = v[v[:, 0].argsort()]

    acc = v[0, 1]
    total_cost = 0

    for i in range(1, len(v)):
        cost_mul = v[i, 0] - v[i - 1, 0]
        total_cost += abs(cost_mul * acc)
        acc += v[i, 1]

    return total_cost

def DySyn_SORD(p_score, n_score, test):
    def f(x):
        return PNTDiff(p_score, n_score, test, x)

    best_alpha = TernarySearch(0, 1, f, 1e-5)  # Implement TernarySearch separately
    vDist = f(best_alpha)

    return [np.array([round(best_alpha, 2), 1 - round(best_alpha, 2)]), vDist]

In [19]:
def ACCSyn(ts, measure, MF_dysyn):
    rQnt = DySyn(ts, measure, MF_dysyn)
    TprFpr = np.array(getTPRandFPRbyThreshold(MoSS(1000, 0.5, rQnt[2]))).astype(float)  # Implement getTPRandFPRbyThreshold

    dC = CC(ts)  # Implement CC function separately

    tpr_fpr_row = TprFpr[TprFpr[:, 0] == 0.5, 1:3].astype(float)
    if tpr_fpr_row.size == 0:
        raise ValueError("Threshold value not found in TprFpr.")

    tpr, fpr = tpr_fpr_row[0]

    result = (dC[0] - fpr) / (tpr - fpr) if (tpr - fpr) != 0 else 0
    result = max(0, min(result, 1))

    return np.array([result, 1 - result], dtype=float)

In [20]:
def XSyn(ts, measure, MF_dysyn):
    rQnt = DySyn(ts, measure, MF_dysyn)
    TprFpr = np.array(getTPRandFPRbyThreshold(MoSS(1000, 0.5, rQnt[2]))).astype(float)
    
    dC = CC(ts)  # Implement CC function separately

    # Usual X implementation
    min_index = abs((1 - TprFpr[:, 1]) - TprFpr[:, 2])
    min_index = np.argmin(min_index)

    tpr_fpr_row = TprFpr[min_index, 1:3].astype(float)
    # if tpr_fpr_row.size == 0:
    #   raise ValueError("Threshold value not found in TprFpr.")

    tpr, fpr = tpr_fpr_row

    result = (dC[0] - fpr) / (tpr - fpr) if (tpr - fpr) != 0 else 0
    result = max(0, min(result, 1))

    return np.array([result, 1 - result], dtype=float)

In [21]:
def MAXSyn(ts, measure, MF_dysyn):
    rQnt = DySyn(ts, measure, MF_dysyn)
    TprFpr = np.array(getTPRandFPRbyThreshold(MoSS(1000, 0.5, rQnt[2]))).astype(float)
    
    dC = CC(ts)  # Implement CC function separately

    # Usual MAX implementation
    max_index = abs(TprFpr[:, 1] - TprFpr[:, 2])
    max_index = np.argmax(max_index)

    tpr_fpr_row = TprFpr[max_index, 1:3].astype(float)

    tpr, fpr = tpr_fpr_row

    result = (dC[0] - fpr) / (tpr - fpr) if (tpr - fpr) != 0 else 0
    result = max(0, min(result, 1))

    return np.array([result, 1 - result], dtype=float)

In [22]:
def T50Syn(ts, measure, MF_dysyn):
    rQnt = DySyn(ts, measure, MF_dysyn)
    TprFpr = np.array(getTPRandFPRbyThreshold(MoSS(1000, 0.5, rQnt[2]))).astype(float)

    dC = CC(ts)  # Implement CC function separately

    # Usual T50 implementation
    min_index = abs(TprFpr[:, 1] - 0.5)
    min_index = np.argmin(min_index)

    tpr_fpr_row = TprFpr[min_index, 1:3].astype(float)

    tpr, fpr = tpr_fpr_row

    result = (dC[0] - fpr) / (tpr - fpr) if (tpr - fpr) != 0 else 0
    result = max(0, min(result, 1))

    return np.array([result, 1 - result], dtype=float)

In [23]:
def MSSyn(ts, measure, MF_dysyn):
    results = []

    rQnt = DySyn(ts, measure, MF_dysyn)
    TprFpr = np.array(getTPRandFPRbyThreshold(MoSS(1000, 0.5, rQnt[2]))).astype(float)

    dC = CC(ts)  # Implement CC function separately

    # Usual MS implementation
    threshold_set = np.arange(0.01, 1.00, 0.01)

    for threshold in threshold_set:
      threshold = round(threshold, 2) # 0.060000003 shenanigans
      tpr, fpr = TprFpr[TprFpr[:, 0] == threshold, 1:3].astype(float)[0]

      result = (dC[0] - fpr) / (tpr - fpr) if (tpr - fpr) != 0 else 0
      result = max(0, min(result, 1))

      results.append(result)

    result = np.median(results)
    return np.array([result, 1 - result], dtype=float)

In [24]:
def MS2Syn(ts, measure, MF_dysyn):
    results = []

    rQnt = DySyn(ts, measure, MF_dysyn)
    TprFpr = np.array(getTPRandFPRbyThreshold(MoSS(1000, 0.5, rQnt[2]))).astype(float)
    
    dC = CC(ts)  # Implement CC function separately

    # Usual MS2 implementation
    index = np.where(abs(TprFpr[:,1]-TprFpr[:,2]) > (1/4))[0].tolist()
    threshold_set = TprFpr[index,0]
    if threshold_set.shape[0] == 0:
      threshold_set = np.arange(0.01, 1.00, 0.01)

    # else usual MS
    for threshold in threshold_set:
      threshold = round(threshold, 2) # 0.060000003 shenanigans
      tpr, fpr = TprFpr[TprFpr[:, 0] == threshold, 1:3].astype(float)[0]

      result = (dC[0] - fpr) / (tpr - fpr) if (tpr - fpr) != 0 else 0
      result = max(0, min(result, 1))

      results.append(result)

    result = np.median(results)
    return np.array([result, 1 - result], dtype=float)

In [25]:
def SMMSyn(ts, measure, MF_dysyn):
    rQnt = DySyn(ts, measure, MF_dysyn)
    best_scores = MoSS(1000, 0.5, rQnt[2]) # sempre NaN em 0 de distances

    best_p = best_scores[best_scores[:, 2] == 1, 0]
    best_n = best_scores[best_scores[:, 2] == 2, 0]
    result = SMM(best_p, best_n, ts)

    return result

# Experiment Real Datasets

## Binarizing experiment between Positive and Negative (Easy and Hard)

In [26]:
experiment_setup

Positive                             ['spec_prior']
Negative    ['priority', 'not_recom', 'very_recom']
Easy                                 ['very_recom']
Hard                      ['priority', 'not_recom']
AUC                                        0.885614
AUC_Easy                                        1.0
AUC_Hard                                   0.883083
diff_AUC                                   0.116917
Name: 47775, dtype: object

In [28]:
df['class'].value_counts()

class
not_recom     4320
priority      4266
spec_prior    4044
very_recom     328
Name: count, dtype: int64

In [29]:
df['binary'] = df['class'].map(lambda x: 'Positive' if x == 'spec_prior' else ('Negative' if x in ['priority', 'not_recom', 'very_recom'] else None))
df['difficulty'] = df['class'].map(lambda x: 'Easy' if x == 'very_recom' else ('Hard' if x in ['priority', 'not_recom'] else 'Positive'))
df['class'] = df['binary']
df.drop(columns=['binary'], inplace=True)
df

,parents,has_nurs,form,children,housing,finance,social,health,class,difficulty
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,Negative,Hard
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,Negative,Hard
4,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,Negative,Hard
5,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,Negative,Hard
6,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,Negative,Hard
...,...,...,...,...,...,...,...,...,...,...
12955,2.0,4.0,3.0,3.0,2.0,1.0,1.0,1.0,Positive,Positive
12956,2.0,4.0,3.0,3.0,2.0,1.0,1.0,2.0,Negative,Hard
12957,2.0,4.0,3.0,3.0,2.0,1.0,2.0,0.0,Positive,Positive
12958,2.0,4.0,3.0,3.0,2.0,1.0,2.0,1.0,Positive,Positive


In [37]:
df['class'].value_counts()

class
Negative    8914
Positive    4044
Name: count, dtype: int64

In [38]:
df[df['difficulty'] == 'Easy'].shape[0], df[df['difficulty'] == 'Hard'].shape[0]

(328, 8586)

## Variables

In [36]:
replicates = 1

test_sizes = [100]
pos_props = np.linspace(0, 1.0, 6)
hard_props = np.linspace(0, 1.0, 6)

quantifiers = []
model = None
pos_props

array([0. , 0.2, 0.4, 0.6, 0.8, 1. ])

## Experiment

### Model training

In [32]:
df['difficulty'].value_counts(), df['difficulty'].value_counts(normalize=True)

(difficulty
 Hard        8586
 Positive    4044
 Easy         328
 Name: count, dtype: int64,
 difficulty
 Hard        0.662602
 Positive    0.312085
 Easy        0.025313
 Name: proportion, dtype: float64)

In [34]:
# TENHO QUE BINARIZAR CLASSES! (FACEIS VIRAM EASY, DIFICEIS VIRAM HARD)

train, test = train_test_split(df, test_size=0.33, stratify=df['difficulty'], random_state=40)


In [37]:
train['difficulty'].value_counts(), train['difficulty'].value_counts(normalize=True), test['difficulty'].value_counts(), test['difficulty'].value_counts(normalize=True)

(difficulty
 Hard        5752
 Positive    2709
 Easy         220
 Name: count, dtype: int64,
 difficulty
 Hard        0.662596
 Positive    0.312061
 Easy        0.025343
 Name: proportion, dtype: float64,
 difficulty
 Hard        2834
 Positive    1335
 Easy         108
 Name: count, dtype: int64,
 difficulty
 Hard        0.662614
 Positive    0.312135
 Easy        0.025251
 Name: proportion, dtype: float64)

In [45]:
minority_class_count = train[train['difficulty'] == 'Easy'].shape[0]
minority_class_count

220

In [ ]:
def get_batch(df, hard_prop):

    df_neg = df[df['class'] == 'Negative']
    df_neg = 
    minority_class_count = 



In [39]:
# TENHO QUE BINARIZAR CLASSES! (FACEIS VIRAM EASY, DIFICEIS VIRAM HARD)

X = df.drop(columns=['class', 'difficulty'])
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, stratify=y, random_state=40)
model = LogisticRegression()

model.fit(X_train, y_train)
y_pred_proba = model.predict_proba(X_test)
y_pred_proba = pd.DataFrame(y_pred_proba, columns=model.classes_)
y_pred_proba

,Negative,Positive
0,0.987338,0.012662
1,0.991244,0.008756
2,0.986853,0.013147
3,0.770417,0.229583
4,0.992955,0.007045
...,...,...
4272,0.947275,0.052725
4273,0.401036,0.598964
4274,0.917272,0.082728
4275,0.558611,0.441389


In [ ]:

for test_size in test_sizes:
    for pos_prop in pos_props:
        for hard_prop in hard_props:
            for replicate in replicates:




                



